In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
import zipfile
from datasets import load_dataset, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel, RobertaModel
from transformers.modeling_outputs import SequenceClassifierOutput
from dataclasses import dataclass
from typing import Optional, Tuple, Any, Dict, List
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda'

In [ ]:
MODEL_NAME = "FacebookAI/roberta-large"
MAX_LENGTH = 512
N_FOLDS = 7

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

evasion_classes = [
    'Declining to answer', 'Dodging', 'General', 'Explicit', 
    'Claims ignorance', 'Clarification', 'Implicit', 
    'Partial/half-answer', 'Deflection'
]
evasion_label2id = {label: idx for idx, label in enumerate(evasion_classes)}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}

In [ ]:
ds = load_dataset("ailsntua/QEvasion")
train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

print(f"Train: {len(train_full_df)}, Test: {len(test_df)}")
print(f"\nEvasion class distribution:")
print(train_full_df['evasion_label'].value_counts())

In [ ]:
y_train = train_full_df['evasion_label'].map(evasion_label2id).values
class_weights = compute_class_weight('balanced', classes=np.array(list(range(9))), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("Class weights:")
for label, weight in zip(evasion_classes, class_weights):
    print(f"  {label}: {weight:.3f}")

In [ ]:
def tokenize_function_train(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=False,
        padding=False,
        max_length=None,
    )
    tokenized["labels"] = [evasion_label2id[l] for l in examples["evasion_label"]]
    return tokenized

def tokenize_function_test(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=False,
        padding=False,
        max_length=None,
    )
    return tokenized

train_full_dataset = Dataset.from_pandas(train_full_df, preserve_index=False)
train_tokenized_full = train_full_dataset.map(
    tokenize_function_train, 
    batched=True, 
    remove_columns=train_full_dataset.column_names
)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_tokenized = test_dataset.map(
    tokenize_function_test, 
    batched=True, 
    remove_columns=test_dataset.column_names
)

In [ ]:
@dataclass
class HierarchicalDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        has_labels = "labels" in features[0]
        
        if has_labels:
            labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
            features_clean = [{k: v for k, v in f.items() if k != "labels"} for f in features]
        else:
            features_clean = features
        
        batch = self.tokenizer.pad(features_clean, padding=True, return_tensors="pt")
        
        if has_labels:
            batch["labels"] = labels
        
        return batch

data_collator = HierarchicalDataCollator(tokenizer=tokenizer)

In [ ]:
class RobertaHierarchicalMaxPooling(RobertaPreTrainedModel):
    def __init__(self, config, class_weights=None):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.class_weights = class_weights
        
        self.roberta = RobertaModel(config)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        
        self.post_init()
    
    def create_chunks_batched(self, input_ids, attention_mask, max_length=512, stride=256):
        batch_size, seq_len = input_ids.shape
        all_chunk_ids = []
        all_chunk_masks = []
        chunk_to_sample = []
        
        for sample_idx in range(batch_size):
            sample_input_ids = input_ids[sample_idx]
            sample_attention_mask = attention_mask[sample_idx]
            actual_length = sample_attention_mask.sum().item()

            if actual_length <= max_length:
                chunk_ids = sample_input_ids[:max_length]
                chunk_mask = sample_attention_mask[:max_length]
                
                if len(chunk_ids) < max_length:
                    padding_length = max_length - len(chunk_ids)
                    chunk_ids = torch.cat([chunk_ids, torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)])
                    chunk_mask = torch.cat([chunk_mask, torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)])
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
            else:
                start = 0
                while start < actual_length:
                    end = min(start + max_length, actual_length)
                    chunk_ids = sample_input_ids[start:end]
                    chunk_mask = sample_attention_mask[start:end]
                    
                    if len(chunk_ids) < max_length:
                        padding_length = max_length - len(chunk_ids)
                        chunk_ids = torch.cat([chunk_ids, torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)])
                        chunk_mask = torch.cat([chunk_mask, torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)])
                    
                    all_chunk_ids.append(chunk_ids)
                    all_chunk_masks.append(chunk_mask)
                    chunk_to_sample.append(sample_idx)
                    start += stride
                    if end >= actual_length:
                        break

        all_chunk_ids = torch.stack(all_chunk_ids, dim=0)
        all_chunk_masks = torch.stack(all_chunk_masks, dim=0)
        chunk_to_sample = torch.tensor(chunk_to_sample, dtype=torch.long, device=input_ids.device)
        return all_chunk_ids, all_chunk_masks, chunk_to_sample
    
    def forward(self, input_ids=None, attention_mask=None, labels=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        batch_size = input_ids.shape[0]

        all_chunk_ids, all_chunk_masks, chunk_to_sample = self.create_chunks_batched(input_ids, attention_mask, max_length=512, stride=256)

        outputs = self.roberta(input_ids=all_chunk_ids, attention_mask=all_chunk_masks, output_attentions=output_attentions, output_hidden_states=output_hidden_states, return_dict=True)

        all_cls_embeddings = outputs.last_hidden_state[:, 0, :]
        batch_embeddings = []
        for sample_idx in range(batch_size):
            sample_mask = chunk_to_sample == sample_idx
            sample_chunk_embeddings = all_cls_embeddings[sample_mask]
            pooled_embedding = torch.max(sample_chunk_embeddings, dim=0)[0]
            batch_embeddings.append(pooled_embedding)

        pooled_output = torch.stack(batch_embeddings, dim=0)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            if self.class_weights is not None:
                weight = self.class_weights.to(logits.device)
                loss_fct = nn.CrossEntropyLoss(weight=weight)
            else:
                loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        if not return_dict:
            output = (logits,)
            return ((loss,) + output) if loss is not None else output
        
        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=None, attentions=None)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1': f1}

class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        if current_metric is None:
            return
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            improved = current_metric > self.best_metric if self.greater_is_better else current_metric < self.best_metric
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
        if self.patience_counter >= self.patience:
            print(f"Early stopping triggered! Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_test_probs = []
fold_oof_probs = []
fold_oof_indices = []
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_full_df, train_full_df['evasion_label']), 1):
    print(f"\nFold {fold}/{N_FOLDS}")
    
    train_dataset = train_tokenized_full.select(train_idx)
    val_dataset = train_tokenized_full.select(val_idx)
    
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=9, id2label=evasion_id2label, label2id=evasion_label2id)
    model = RobertaHierarchicalMaxPooling(config, class_weights=class_weights_tensor)
    
    base_model = AutoModel.from_pretrained(MODEL_NAME)
    model.roberta.load_state_dict(base_model.state_dict())
    del base_model

    training_args = TrainingArguments(
        output_dir=f"./results_roberta_evasion_fold{fold}",
        learning_rate=1e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=15,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        bf16=True,
        bf16_full_eval=True,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED + fold,
        report_to="none",
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1")],
    )
    
    trainer.train()
    
    val_results = trainer.evaluate()
    fold_metrics.append({'fold': fold, 'val_accuracy': val_results['eval_accuracy'], 'val_f1': val_results['eval_f1']})
    print(f"Fold {fold}: Acc={val_results['eval_accuracy']:.4f}, F1={val_results['eval_f1']:.4f}")
    
    oof_output = trainer.predict(val_dataset)
    oof_probs = torch.nn.functional.softmax(torch.tensor(oof_output.predictions), dim=-1).numpy()
    fold_oof_probs.append(oof_probs)
    fold_oof_indices.append(val_idx)
    
    test_output = trainer.predict(test_tokenized)
    test_probs = torch.nn.functional.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()
    fold_test_probs.append(test_probs)
    
    del model, trainer
    torch.cuda.empty_cache()

In [ ]:
metrics_df = pd.DataFrame(fold_metrics)
print(metrics_df.to_string(index=False))
print(f"\nAvg Accuracy: {metrics_df['val_accuracy'].mean():.4f} ± {metrics_df['val_accuracy'].std():.4f}")
print(f"Avg Macro F1: {metrics_df['val_f1'].mean():.4f} ± {metrics_df['val_f1'].std():.4f}")

oof_probs_full = np.zeros((len(train_full_df), 9))
for probs, idxs in zip(fold_oof_probs, fold_oof_indices):
    oof_probs_full[idxs] = probs

oof_preds = np.argmax(oof_probs_full, axis=1)
oof_pred_labels = [evasion_id2label[p] for p in oof_preds]
y_true_train = train_full_df['evasion_label'].tolist()

print("\nOOF Classification Report:")
print(classification_report(y_true_train, oof_pred_labels, labels=evasion_classes, zero_division=0))

In [ ]:
ensemble_probs = np.mean(fold_test_probs, axis=0)
ensemble_preds = np.argmax(ensemble_probs, axis=1)
ensemble_labels = [evasion_id2label[p] for p in ensemble_preds]

y_true_test = test_df['evasion_label'].tolist()

print(f"Test Accuracy: {accuracy_score(y_true_test, ensemble_labels):.4f}")
print(f"Test Macro F1: {f1_score(y_true_test, ensemble_labels, average='macro', labels=evasion_classes):.4f}")
print("\nTest Classification Report:")
print(classification_report(y_true_test, ensemble_labels, labels=evasion_classes, zero_division=0))

In [ ]:
def compute_relaxed_accuracy(predictions, test_dataframe, id2label_mapping):
    pred_labels = [id2label_mapping[p] for p in predictions]
    correct_count = 0
    total_count = len(predictions)
    
    for idx, pred_label in enumerate(pred_labels):
        annotator1 = test_dataframe.iloc[idx].get('annotator1', None)
        annotator2 = test_dataframe.iloc[idx].get('annotator2', None)
        annotator3 = test_dataframe.iloc[idx].get('annotator3', None)
        
        if pred_label in [annotator1, annotator2, annotator3]:
            correct_count += 1
    
    return correct_count / total_count if total_count > 0 else 0.0

annotator_cols = ['annotator1', 'annotator2', 'annotator3']
has_annotators = all(col in test_df.columns for col in annotator_cols)

if has_annotators:
    relaxed_acc = compute_relaxed_accuracy(ensemble_preds, test_df, evasion_id2label)
    standard_acc = accuracy_score(y_true_test, ensemble_labels)
    print(f"Relaxed Accuracy: {relaxed_acc:.4f}")
    print(f"Standard Accuracy: {standard_acc:.4f}")
    print(f"Improvement: {(relaxed_acc - standard_acc)*100:.2f}%")
else:
    print("Annotator columns not found")

In [ ]:
os.makedirs("./predictions", exist_ok=True)
os.makedirs("../predictions", exist_ok=True)

prediction_filename = "./predictions/prediction"
with open(prediction_filename, 'w') as f:
    for label in ensemble_labels:
        f.write(f"{label}\n")

zip_filename = "../predictions/prediction_evasion_single_head_kfold.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(prediction_filename, arcname="prediction")

print(f"Saved to {zip_filename}")